In [ ]:
import sys
from pathlib import Path
# Add the parent directory to sys.path so we can import synth_extract
sys.path.insert(0, str(Path.cwd().parent)) 

In [ ]:
from synth_extract.agents.classification import (  # noqa: E402
    ClassificationFailure,
    ClassificationResult,
    ClassificationOutcome,
    PaperClassifier,
    FullTextClassifier
)
from synth_extract.agents.llm import LLMBackend  # noqa: E402

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import asyncio

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

In [ ]:
host="127.0.0.1"
port="8000"
base_url=f"http://{host}:{port}/v1"

model = "qwen3.6-27b"
api_key = "none"

max_tokens=8192
extra_body = {"chat_template_kwargs":{"enable_thinking":True}}

backend = LLMBackend(
    model=model,
    base_url=base_url,
    api_key=api_key,
    temperature=0.0,
    timeout=300,
    max_tokens=max_tokens,
    extra_body=extra_body,
)

In [ ]:
base_path = Path("/nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract/del_classification")
system_prompt_path = base_path / "s1.md"
user_prompt_path = base_path / "user_prompt.md"

In [ ]:
classifier = FullTextClassifier(backend=backend,
system_prompt_path=system_prompt_path,
user_template_path=user_prompt_path
)

In [ ]:
classifier.health_check()

In [ ]:
dev_path = Path("/nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract/data/development_set")
label_path = dev_path / "dataset_labels.csv"

In [ ]:
df = pd.read_csv(label_path)

In [ ]:
df.head()

In [ ]:
def dataset_stats(data, name):
    total = len(data)
    positives = (data["label"] == 1).sum()
    negatives = (data["label"] == 0).sum()

    uncertain = data["uncertain"].notna().sum()
    certain = data["uncertain"].isna().sum()

    certain_data = data[data["uncertain"].isna()]
    certain_positives = (certain_data["label"] == 1).sum()
    certain_negatives = (certain_data["label"] == 0).sum()

    return {
        "Dataset": name,
        "N": total,
        
        "Positive": positives,
        "Negative": negatives,
        "Uncertain": uncertain,
        "Certain": certain,
        "N after removing uncertain": len(certain_data),
        "Positive after removing uncertain": certain_positives,
        "Negative after removing uncertain": certain_negatives,
    }


stats = [
    dataset_stats(df, "Full"),
    dataset_stats(df[df["split"] == "train"], "Train"),
    dataset_stats(df[df["split"] == "test"], "Test"),
]

dataset_stats_table = pd.DataFrame(stats).set_index("Dataset")

dataset_stats_table

In [ ]:
# df = df[df["canonical_source"].isin(["wiley", "springer_nature"])]
# print(f"Number of rows after filtering: {len(df)}")

In [ ]:
df

In [ ]:
system_prompt_path = base_path / "s5.1.md"
classifier.update_prompt_paths(system_prompt_path=system_prompt_path)

In [ ]:
print(classifier.system_prompt())

In [ ]:
classifier.backend.extra_body = {"chat_template_kwargs":{"enable_thinking":True}}

In [ ]:
classifier.backend.config()

In [ ]:
# # Pick a paper
# uid = "ID000726529"

# # Find its row
# row = df.loc[df["paper_uid"] == uid].iloc[0]

# source = row["canonical_source"]

# # Construct full-text path
# fulltext_path = (
#     Path(dev_path)
#     / source
#     / uid
#     / f"{uid}.md"
# )

# print("Source:", source)
# print("Full text:", fulltext_path)

# if not fulltext_path.exists():
#     raise FileNotFoundError(fulltext_path)

# # Load Markdown
# full_text = fulltext_path.read_text(encoding="utf-8")

# print(f"Characters: {len(full_text):,}")

# # Classify
# result = classifier.classify(full_text)

# result

In [ ]:
# print(result.metadata.reasoning)

In [ ]:
max_parallel_requests = 8
async_semaphore = asyncio.Semaphore(max_parallel_requests)


async def classify_row_async(
    n,
    row,
) -> ClassificationResult | ClassificationFailure | None:
    uid = row["paper_uid"]
    source = row["canonical_source"]
    true_label = int(row["label"])
    fulltext_path = Path(dev_path) / source / uid / f"{uid}.md"

    if not fulltext_path.exists():
        print(
            f"[{n}/{len(df)}] Source: {source} | UID: {uid} | "
            f"True: {true_label} | Prediction: None | "
            f"Tokens: None | Missing: {fulltext_path}"
        )
        return None

    try:
        full_text = fulltext_path.read_text(encoding="utf-8")

        async with async_semaphore:
            result = await classifier.aclassify(full_text)

        if isinstance(result, ClassificationResult):
            prediction = int(result.label)
            usage = result.metadata.usage

            token_summary = (
                f"input={usage.prompt_tokens}, "
                f"output={usage.completion_tokens}, "
                f"total={usage.total_tokens}"
                if usage is not None
                else "unavailable"
            )

            print(
                f"[{n}/{len(df)}] Source: {source} | UID: {uid} | "
                f"True: {true_label} | Prediction: {prediction} | "
                f"Tokens: {token_summary}"
            )
            return result

        if isinstance(result, ClassificationFailure):
            print(
                f"[{n}/{len(df)}] Source: {source} | UID: {uid} | "
                f"True: {true_label} | Prediction: None | "
                f"Tokens: None | Failure: "
                f"{result.error_type}: {result.message}"
            )
            return result

        raise TypeError(
            f"Unexpected classification result: {type(result).__name__}"
        )

    except Exception as exc:
        print(
            f"[{n}/{len(df)}] Source: {source} | UID: {uid} | "
            f"True: {true_label} | Prediction: None | "
            f"Tokens: None | Error: {type(exc).__name__}: {exc}"
        )
        return None


tasks = [
    asyncio.create_task(classify_row_async(n, row))
    for n, (_, row) in enumerate(df.iterrows(), start=1)
]

classification_outcomes: list[
    ClassificationResult | ClassificationFailure | None
] = await asyncio.gather(*tasks)

In [ ]:
# Evaluate only successful classifications
valid = [
    isinstance(outcome, ClassificationResult)
    for outcome in classification_outcomes
]

y_true = df.loc[valid, "label"].astype(int)
y_pred = [
    int(outcome.label)
    for outcome in classification_outcomes
    if isinstance(outcome, ClassificationResult)
]

classification_failures = sum(
    isinstance(outcome, ClassificationFailure)
    for outcome in classification_outcomes
)
missing_or_errors = sum(
    outcome is None
    for outcome in classification_outcomes
)
print(f"Actual negatives:        {(y_true == 0).sum()}")
print(f"Actual positives:        {(y_true == 1).sum()}")

print(f"Evaluated:               {len(y_pred)}/{len(classification_outcomes)}")
print(f"Classification failures: {classification_failures}")
print(f"Missing/errors:          {missing_or_errors}")
print()

print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
print(f"Precision: {precision_score(y_true, y_pred):.3f}")
print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
print(f"F1:        {f1_score(y_true, y_pred):.3f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

In [ ]:
# Evaluate only successful classifications for non-uncertain papers
valid = [
    isinstance(outcome, ClassificationResult)
    for outcome in classification_outcomes
]

non_uncertain = df["uncertain"].isna().to_numpy()

eval_mask = [
    is_valid and is_non_uncertain
    for is_valid, is_non_uncertain in zip(valid, non_uncertain)
]

y_true = df.loc[eval_mask, "label"].astype(int)

y_pred = [
    int(outcome.label)
    for outcome, keep in zip(classification_outcomes, eval_mask)
    if keep
]

classification_failures = sum(
    isinstance(outcome, ClassificationFailure) and is_non_uncertain
    for outcome, is_non_uncertain in zip(classification_outcomes, non_uncertain)
)

missing_or_errors = sum(
    outcome is None and is_non_uncertain
    for outcome, is_non_uncertain in zip(classification_outcomes, non_uncertain)
)

print("Non-uncertain papers only")
print(f"Actual negatives:        {(y_true == 0).sum()}")
print(f"Actual positives:        {(y_true == 1).sum()}")

print(f"Evaluated:               {len(y_pred)}/{non_uncertain.sum()}")
print(f"Classification failures: {classification_failures}")
print(f"Missing/errors:          {missing_or_errors}")
print()

print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
print(f"Precision: {precision_score(y_true, y_pred):.3f}")
print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
print(f"F1:        {f1_score(y_true, y_pred):.3f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

In [ ]:
for split_name in ["train", "test"]:
    split_mask = df["split"] == split_name

    valid = [
        split_mask.iloc[i] and isinstance(outcome, ClassificationResult)
        for i, outcome in enumerate(classification_outcomes)
    ]

    y_true = df.loc[valid, "label"].astype(int)

    y_pred = [
        int(outcome.label)
        for i, outcome in enumerate(classification_outcomes)
        if split_mask.iloc[i] and isinstance(outcome, ClassificationResult)
    ]

    classification_failures = sum(
        split_mask.iloc[i] and isinstance(outcome, ClassificationFailure)
        for i, outcome in enumerate(classification_outcomes)
    )

    missing_or_errors = sum(
        split_mask.iloc[i] and outcome is None
        for i, outcome in enumerate(classification_outcomes)
    )

    total = split_mask.sum()

    print("=" * 60)
    print(split_name.upper())
    print("=" * 60)

    print(f"Actual negatives:        {(y_true == 0).sum()}")
    print(f"Actual positives:        {(y_true == 1).sum()}")

    print(f"Evaluated:               {len(y_pred)}/{total}")
    print(f"Classification failures: {classification_failures}")
    print(f"Missing/errors:           {missing_or_errors}")
    print()

    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred):.3f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
    print(f"F1:        {f1_score(y_true, y_pred):.3f}")

    print("\nConfusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print()

In [ ]:
results = {}

for scope_name, scope_mask in {
    "Full": pd.Series(True, index=df.index),
    "Certain only": df["uncertain"].isna(),
}.items():

    for split_name in ["train", "test"]:
        split_mask = df["split"] == split_name
        mask = scope_mask & split_mask

        valid = [
            mask.iloc[i] and isinstance(outcome, ClassificationResult)
            for i, outcome in enumerate(classification_outcomes)
        ]

        y_true = df.loc[valid, "label"].astype(int)

        y_pred = [
            int(outcome.label)
            for i, outcome in enumerate(classification_outcomes)
            if mask.iloc[i] and isinstance(outcome, ClassificationResult)
        ]

        results[(scope_name, split_name.capitalize())] = {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
        }

metrics_table = pd.DataFrame(results)

# Optional: round for cleaner display
metrics_table = metrics_table.round(3)

metrics_table

In [ ]:
valid = [
    isinstance(outcome, ClassificationResult)
    for outcome in classification_outcomes
]

y_true = df.loc[valid, "label"].astype(int)
y_pred = [
    int(outcome.label)
    for outcome in classification_outcomes
    if isinstance(outcome, ClassificationResult)
]

failed = df.loc[valid, ["paper_uid", "canonical_source", "title", "split", "label", "uncertain"]].copy()
failed["pred_label"] = y_pred

failed = failed[
    failed["label"].astype(int) != failed["pred_label"]
]

failed

In [ ]:
failed[(failed["split"] == "train") & (failed["uncertain"] != 1)]

In [ ]:
for i, outcome in enumerate(classification_outcomes):
    row = df.iloc[i]

    if row["split"] != "train":
        continue

    if not isinstance(outcome, ClassificationResult):
        continue

    if row["uncertain"] == 1:
        continue

    true_label = int(row["label"])
    pred_label = int(outcome.label)

    if true_label == pred_label:
        continue

    print("=" * 100)
    print("UID:       ", row["paper_uid"])
    print("Title:     ", row["title"])
    print("True:      ", true_label)
    print("Predicted: ", pred_label)

    error_type = "FP" if true_label == 0 and pred_label == 1 else "FN"
    print("Error type: ", error_type)

    if outcome.metadata.reasoning:
        print("\nReasoning:")
        print(outcome.metadata.reasoning)

    print()

In [ ]:
for i, outcome in enumerate(classification_outcomes):
    row = df.iloc[i]

    if row["split"] != "train":
        continue

    if not isinstance(outcome, ClassificationResult):
        continue

    if row["uncertain"] == 1:
        continue

    true_label = int(row["label"])
    pred_label = int(outcome.label)

    if pred_label == 0:
        continue

    print("=" * 100)
    print("UID:       ", row["paper_uid"])
    print("Title:     ", row["title"])
    print("Predicted: ", pred_label)

    if outcome.metadata.reasoning:
        print("\nReasoning:")
        print(outcome.metadata.reasoning)

    print()

In [ ]:
# ### Update the table

# column_name = "qwen_s5.1_res"
# df[column_name] = pd.array(
#     [
#         int(outcome.label)
#         if isinstance(outcome, ClassificationResult)
#         else pd.NA
#         for outcome in classification_outcomes
#     ],
#     dtype="Int64",
# )

# # label_path = dev_path / "dataset_labels_del.csv"
# df.to_csv(label_path, index=False)